In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import glob

from torch_geometric.data import Data

In [2]:
#import torch_geometric

TORCH = torch.__version__.split('+')[0]
CUDA = 'cu' + torch.version.cuda.replace('.','')

#!pip install torch-scatter     -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
#!pip install torch-sparse      -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
#!pip install torch-cluster     -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
#!pip install torch-spline-conv -f https://pytorch-geometric.com/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric 
import torch_geometric
import torch_geometric.nn as geom_nn
import torch_geometric.data as geom_data

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import torch_geometric
#it works despite all those errors

In [161]:
pdb1=pd.read_csv('../../dsci410_510/data/Fitzgerald_ens/ens_060/ens_060_pdb1_6IBD.csv')
pdb1[0:60]

,resid_num,wildtype,atom,x,y,z
0,1,P,N,-50.351,23.505,-11.865
1,1,P,CA,-50.667,22.127,-12.330
2,1,P,C,-49.461,21.163,-12.284
3,1,P,O,-48.412,21.481,-12.828
4,1,P,CB,-51.145,22.322,-13.772
5,1,P,CG,-50.279,23.466,-14.261
6,1,P,CD,-50.039,24.341,-13.042
7,2,E,N,-49.637,20.022,-11.618
8,2,E,CA,-48.540,19.093,-11.231
9,2,E,C,-48.117,18.241,-12.420


In [162]:
pdb1[45:80]

,resid_num,wildtype,atom,x,y,z
45,6,I,CG2,-40.737,16.109,-13.425
46,6,I,CD1,-38.511,14.295,-12.119
47,7,T,N,-38.089,15.442,-16.825
48,7,T,CA,-36.837,15.035,-17.481
49,7,T,C,-35.661,15.226,-16.537
50,7,T,O,-35.585,16.174,-15.750
51,7,T,CB,-36.593,15.705,-18.829
52,7,T,OG1,-36.178,17.048,-18.667
53,7,T,CG2,-37.797,15.617,-19.751
54,8,I,N,-34.797,14.207,-16.541


In [5]:
pdb2=pd.read_csv('../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_pdb2_6QEB.csv')
pdb2

,resid_num,wildtype,atom,x,y,z
0,1,M,N,26.177,13.897,62.007
1,1,M,CA,25.040,13.829,61.052
2,1,M,C,24.705,15.212,60.506
3,1,M,O,24.407,15.367,59.322
4,1,M,CB,23.809,13.236,61.741
...,...,...,...,...,...,...
2068,260,K,CG,-0.529,-8.342,35.808
2069,260,K,CD,-1.758,-9.196,36.072
2070,260,K,CE,-1.458,-10.310,37.061
2071,260,K,NZ,-2.642,-11.184,37.289


In [6]:
ddg_labels=pd.read_csv('../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_max_ddgs.csv')
ddg_labels

,resid_num,wildtype,max_diff_ddg
0,1.0,M,0.641229
1,2.0,S,0.252171
2,3.0,H,0.154710
3,4.0,H,1.733517
4,5.0,W,1.594987
...,...,...,...
255,256.0,K,2.137001
256,257.0,A,1.853797
257,258.0,S,2.218604
258,259.0,F,0.598084


In [8]:
pdb1

,resid_num,wildtype,atom,x,y,z
0,1,M,N,30.647,-1.764,-0.233
1,1,M,CA,31.902,-1.036,-0.539
2,1,M,C,32.172,0.185,0.353
3,1,M,O,33.340,0.531,0.508
4,1,M,CB,31.930,-0.603,-2.012
...,...,...,...,...,...,...
2068,260,K,CG,-3.542,-5.157,37.232
2069,260,K,CD,-4.735,-5.131,38.194
2070,260,K,CE,-4.960,-3.722,38.743
2071,260,K,NZ,-6.264,-3.518,39.424


In [7]:
for item in pdb1.resid_num:
    mydf=pdb1.loc[pdb1.resid_num==item]

print(mydf.loc[:,['atom','x','y','z']])

     atom      x      y       z
2063    N -2.185 -5.800  34.646
2064   CA -2.017 -6.671  35.819
2065    C -1.831 -8.114  35.399
2066    O -2.319 -8.528  34.345
2067   CB -3.231 -6.566  36.741
2068   CG -3.542 -5.157  37.232
2069   CD -4.735 -5.131  38.194
2070   CE -4.960 -3.722  38.743
2071   NZ -6.264 -3.518  39.424
2072  OXT -1.215 -8.906  36.123


In [9]:
np.array(mydf.loc[:,['x','y','z']])

array([[-2.185, -5.8  , 34.646],
       [-2.017, -6.671, 35.819],
       [-1.831, -8.114, 35.399],
       [-2.319, -8.528, 34.345],
       [-3.231, -6.566, 36.741],
       [-3.542, -5.157, 37.232],
       [-4.735, -5.131, 38.194],
       [-4.96 , -3.722, 38.743],
       [-6.264, -3.518, 39.424],
       [-1.215, -8.906, 36.123]])

it's easy enough to get a graph of coordinates for the x y z coordinates
however, I want info on the atoms--could I one hot encode the atoms?

also I want info on the residue itself--should I also one hot encode that? 

so that would result in a dataframe with the site number, site identity, list of atom identities, and array of coordinate values for each of those identities

maybe start there and then figure out how to encode them and whatnot

In [10]:
site=[]
resid=[]
atom_list=[]
coord_array=[]

for item in pdb1.resid_num.unique():
    mydf=pdb1.loc[pdb1.resid_num==item]

    site.append(mydf.resid_num.to_list()[0])
    resid.append(mydf.wildtype.to_list()[0])
    atom_list.append(mydf.atom.to_list())
    coord_array.append(np.array(mydf.loc[:,['x','y','z']]))

df=pd.DataFrame({'site':site,'resid':resid,'atom_list':atom_list,'coord_array':coord_array})
    

In [11]:
df

,site,resid,atom_list,coord_array
0,1,M,"[N, CA, C, O, CB, CG, SD, CE]","[[30.647, -1.764, -0.233], [31.902, -1.036, -0..."
1,2,S,"[N, CA, C, O, CB, OG]","[[31.133, 0.855, 0.893], [31.362, 2.073, 1.731..."
2,3,H,"[N, CA, C, O, CB, CG, ND1, CD2, CE1, NE2]","[[32.407, 2.694, 3.825], [32.693, 2.55, 5.296]..."
3,4,H,"[N, CA, C, O, CB, CG, ND1, CD2, CE1, NE2]","[[31.019, 1.843, 6.877], [29.793, 1.99, 7.618]..."
4,5,W,"[N, CA, C, O, CB, CG, CD1, CD2, NE1, CE2, CE3,...","[[29.017, 1.393, 9.842], [28.6, 0.403, 10.866]..."
...,...,...,...,...
255,256,K,"[N, CA, C, O, CB, CG, CD, CE, NZ]","[[8.044, -0.995, 33.656], [6.682, -0.862, 34.2..."
256,257,A,"[N, CA, C, O, CB]","[[4.701, -2.169, 33.77], [3.591, -2.746, 33.01..."
257,258,S,"[N, CA, C, O, CB, OG]","[[1.598, -1.793, 32.022], [0.354, -1.03, 32.01..."
258,259,F,"[N, CA, C, O, CB, CG, CD1, CD2, CE1, CE2, CZ]","[[-0.577, -3.144, 32.739], [-1.598, -4.16, 32...."


In [ ]:
class EnsDataset(Dataset):

    def __init__(self, root_dir, transform=None,target_transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform ## I guess I will keep these? in case I am doing any transforming
        self.target_transform = target_transform 

        self.paths=paths
        self.labels= labels

        #look at lab 3 for examples
        #everything under here is just an example
        labels=[]
        paths=[]
        for i in glob.glob(root_dir+'/*/*'):
            paths.append(i)
            #labels.append(i)
            num=int(((i.split('/')[-1].split('_')[0]))[1])
            if num in [1,2,3]:
                labels.append(num-1)
            if num in [5,6]:
                labels.append(num-2)
        self.img_labels = labels
        self.img_paths = paths
    
    def __len__(self):
        # Returns the number of samples
        return len(self.img_labels)


    def __getitem__(self, idx):

        # Here we need to use index to:
        # 1. grab the corresponding image and label
        # 2. run any transforms on the image
        # 3. return the transformed image and label as tensors #this part feels very important to me
        img_path=self.img_paths[idx]
        image=Image.open(img_path)
        label=self.img_labels[idx]
        if self.transform:
            image=self.transform(image)
        if self.target_transform:
            label=self.target_transform(label)
        return image, label



In [12]:
atom_dict={'C':1, 'N':2,'O':3,'S':4}


In [9]:
aa=['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']

In [10]:
aa_dict= dict([(aa[i], i) for i in range(0,len(aa))])

In [11]:
aa_dict

{'A': 0,
 'C': 1,
 'D': 2,
 'E': 3,
 'F': 4,
 'G': 5,
 'H': 6,
 'I': 7,
 'K': 8,
 'L': 9,
 'M': 10,
 'N': 11,
 'P': 12,
 'Q': 13,
 'R': 14,
 'S': 15,
 'T': 16,
 'V': 17,
 'W': 18,
 'Y': 19}

In [116]:
aa_dict.get('D')

2

In [8]:
atom_dict

{'C': 1, 'N': 2, 'O': 3, 'S': 4}

In [ ]:
from torch_geometric.data import Data

In [10]:
data=Data(

torch_geometric.data.data.Data

In [2]:
print('hi')

hi


In [4]:
ens_dir_list=['../../dsci410_510/data/Fitzgerald_ens/ens_004/','../../dsci410_510/data/Fitzgerald_ens/ens_060/',
              '../../dsci410_510/data/Fitzgerald_ens/ens_098/']

In [11]:
pdb_files=[]
for i in ens_dir_list:
    pdb_files.append(glob.glob(i+'*pdb*csv'))

so I need to figure out at what point I combine pdb1 and pdb2 and how that influences how I load my data
could I have some kind of label that labels my data as per ens
or do I load all pdb1 files and pdb2 files together and then have those run partially before comparing them
I suppose one option is to include two coordinates as different node features?
for now, just figure out code for learning edge features

In [12]:
pdb1=pd.read_csv('../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_pdb1_5JQT.csv')

In [13]:
site=[]
resid=[]
atom_list=[]
coord_array=[]

CA_only=pdb1.loc[pdb1['atom']=='CA']

for idx,rows in CA_only.iterrows():

    site.append(CA_only.loc[idx,'resid_num'])
    resid.append(CA_only.loc[idx,'wildtype'])
    coord_array.append(np.array(CA_only.loc[idx,['x','y','z']]))


df=pd.DataFrame({'site':site,'resid':resid,'coord_array':coord_array})

In [95]:
df

,site,resid,coord_array
0,1,M,"[31.902, -1.036, -0.539]"
1,2,S,"[31.362, 2.073, 1.731]"
2,3,H,"[32.693, 2.55, 5.296]"
3,4,H,"[29.793, 1.99, 7.618]"
4,5,W,"[28.6, 0.403, 10.866]"
...,...,...,...
255,256,K,"[6.682, -0.862, 34.212]"
256,257,A,"[3.591, -2.746, 33.011]"
257,258,S,"[0.354, -1.03, 32.017]"
258,259,F,"[-1.598, -4.16, 32.979]"


In [31]:
len(df)

260

In [23]:
np.linalg.norm(np.array([31.902, -1.036, -0.539])-np.array([31.362, 2.073, 1.731]))

3.887207352328919

In [40]:
#pdb file coordinate numbers are angstroms
#lets do all carbon alpha values within 10a of each other
edges=[]
edge_features=[]
#0 for near each other, 1 for backbone bond
for idx1,rows1 in df.iterrows():

    p1=df.loc[idx1,'coord_array']

    if idx1 == 0:
        edges.append((idx1,idx1+1))
        edge_features.append(1)

    elif idx1 == len(df)-1:
        edges.append((idx1,idx1-1))
        edge_features.append(1)
        
    else:
        edges.extend([(idx1, idx1-1),(idx1,idx1+1)])
        edge_features.extend([1,1])

    for idx2,rows2 in df.iterrows():

        if idx1 != idx2:

            p2=df.loc[idx2,'coord_array']
    
            d = np.linalg.norm(p1 - p2)

            if d<=10:

                #do I assume that the nodes start at 0 and end at n number of nodes? 
                #if so, I will use the index rather than the resid_num
                #for now, will make that assumption

                edges.append((idx1,idx2))
                edge_features.append(0)
        
        

In [42]:
print(edge_features)

[1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [96]:
#need to modify this function to get edge info for a bunch of graphs added together
#but need to make sure those nodes are not talking to each other
def get_edge_info(df):
    
    #pdb file coordinate numbers are angstroms
    #lets do all carbon alpha values within 10a of each other
    edges=[]
    edge_features=[]
    #0 for near each other, 1 for backbone bond
    for idx1,rows1 in df.iterrows():
    
        p1=df.loc[idx1,'coord_array']
    
        if idx1 == 0:
            edges.append((idx1,idx1+1))
            edge_features.append(1)
    
        elif idx1 == len(df)-1:
            edges.append((idx1,idx1-1))
            edge_features.append(1)
            
        else:
            edges.extend([(idx1, idx1-1),(idx1,idx1+1)])
            edge_features.extend([1,1])
    
        for idx2,rows2 in df.iterrows():
    
            if idx1 != idx2:
    
                p2=df.loc[idx2,'coord_array']
        
                d = np.linalg.norm(p1 - p2)
    
                if d<=10:
    
                    #do I assume that the nodes start at 0 and end at n number of nodes? 
                    #if so, I will use the index rather than the resid_num
                    #for now, will make that assumption
    
                    edges.append((idx1,idx2))
                    edge_features.append(0)

    return edges, edge_features
            
            
        

In [97]:
get_edge_info(df)

([(0, 1),
  (0, 1),
  (0, 2),
  (0, 3),
  (0, 230),
  (0, 231),
  (0, 234),
  (0, 235),
  (1, 0),
  (1, 2),
  (1, 0),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 230),
  (1, 231),
  (2, 1),
  (2, 3),
  (2, 0),
  (2, 1),
  (2, 3),
  (2, 4),
  (2, 5),
  (2, 10),
  (3, 2),
  (3, 4),
  (3, 0),
  (3, 1),
  (3, 2),
  (3, 4),
  (3, 5),
  (3, 6),
  (3, 10),
  (3, 11),
  (3, 62),
  (3, 63),
  (4, 3),
  (4, 5),
  (4, 1),
  (4, 2),
  (4, 3),
  (4, 5),
  (4, 6),
  (4, 7),
  (4, 9),
  (4, 10),
  (4, 11),
  (4, 12),
  (4, 14),
  (4, 15),
  (4, 18),
  (4, 62),
  (4, 63),
  (4, 198),
  (4, 199),
  (5, 4),
  (5, 6),
  (5, 2),
  (5, 3),
  (5, 4),
  (5, 6),
  (5, 7),
  (5, 8),
  (5, 9),
  (5, 10),
  (5, 11),
  (5, 12),
  (5, 14),
  (5, 15),
  (5, 62),
  (5, 63),
  (5, 198),
  (5, 199),
  (5, 229),
  (5, 242),
  (5, 243),
  (6, 5),
  (6, 7),
  (6, 3),
  (6, 4),
  (6, 5),
  (6, 7),
  (6, 8),
  (6, 9),
  (6, 10),
  (6, 11),
  (6, 12),
  (6, 13),
  (6, 14),
  (6, 15),
  (6, 63),
  (6, 198),
  (6, 229),
  (6, 239),
 

In [69]:
ens_dir_list=['../../dsci410_510/data/Fitzgerald_ens/ens_004/','../../dsci410_510/data/Fitzgerald_ens/ens_060/',
              '../../dsci410_510/data/Fitzgerald_ens/ens_098/']

In [54]:
pdb_files=[]
for i in ens_dir_list:
    pdb_files.append(glob.glob(i+'*csv'))

In [55]:
pdb_files

[['../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_pdb2_6QEB.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_pdb1_5JQT.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_004/ens_004_max_ddgs.csv'],
 ['../../dsci410_510/data/Fitzgerald_ens/ens_060/ens_060_pdb2_5RY1.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_060/ens_060_pdb1_6IBD.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_060/ens_060_max_ddgs.csv'],
 ['../../dsci410_510/data/Fitzgerald_ens/ens_098/ens_098_pdb2_7QBO.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_098/ens_098_pdb1_7QBN.csv',
  '../../dsci410_510/data/Fitzgerald_ens/ens_098/ens_098_max_ddgs.csv']]

In [83]:
def old_prep_for_graph_features(df):
    
    site=[]
    resid=[]
    #ideally resid is one hot encoded
    atom_list=[]
    coord_array=[]
    num_atoms=[]
    #these do not need to be one hot encoded
    
    CA_only=df.loc[df['atom']=='CA']
    
    for idx,rows in CA_only.iterrows():
    
        site.append(CA_only.loc[idx,'resid_num'])
        resid.append(CA_only.loc[idx,'wildtype'])
        coord_array.append(np.array(CA_only.loc[idx,['x','y','z']]))
    
    
    prepped_df=pd.DataFrame({'site':site,'resid':resid,'coord_array':coord_array})

    return prepped_df

In [5]:
def prep_for_graph_features(df):
#if I have time later, I would consider going in and loading in coordinates for all atoms
#but I probably will not tbh
    
    site=[]
    resid=[]  #this will be one hot encoded later
    coord_array=[]
    x=[]
    y=[]
    z=[]
    num_C=[]
    num_N=[]
    num_O=[]
    num_S=[]

    for item in df.resid_num.unique():
        mydf=df.loc[df.resid_num==item]

        atom_list=mydf.atom.to_list()

        just_elem=[i[0] for i in mydf.atom.to_list()]
        num_C.append(just_elem.count('C'))
        num_N.append(just_elem.count('N'))
        num_O.append(just_elem.count('O'))
        num_S.append(just_elem.count('S'))

                     
        site.append(mydf.resid_num.to_list()[0])
        resid.append(aa_dict.get(mydf.wildtype.to_list()[0]))
        atom_list.append(mydf.atom.to_list())
        
        CA_only=mydf.loc[df['atom']=='CA']

        coord_array.append(np.array(CA_only.loc[:,['x','y','z']]))
        x.append(CA_only.loc[:,'x'].to_list()[0])
        y.append(CA_only.loc[:,'y'].to_list()[0])
        z.append(CA_only.loc[:,'z'].to_list()[0])
                     
    
    prepped_df=pd.DataFrame({'site':site,
                             'resid':resid,
                             'coord_array':coord_array,
                             'x_coord':x,
                             'y_coord':y,
                             'z_coord':z,
                             'num_C':num_C,
                             'num_N':num_N,
                             'num_O':num_O,
                             'num_S':num_S})

    return prepped_df

In [6]:
def combining_ensembles(ens_dir_list):
    
    pdb1_df=pd.DataFrame()
    pdb2_df=pd.DataFrame()
    ddg_df=pd.DataFrame()
    
    for i in ens_dir_list:
    
        this_ens=i.split('/')[-2]
        
        df1=prep_for_graph_features(pd.read_csv(glob.glob(i+'*pdb1*csv')[0]))
        df1['ens']=this_ens
        pdb1_df=pd.concat([pdb1_df,df1],ignore_index=True)
        
        df2=prep_for_graph_features(pd.read_csv(glob.glob(i+'*pdb2*csv')[0]))
        df2['ens']=this_ens
        pdb2_df=pd.concat([pdb2_df,df2],ignore_index=True)
        
    
        ddgs=pd.read_csv(glob.glob(i+'*ddgs.csv')[0])
        ddgs['ens']=this_ens
        ddg_df=pd.concat([ddg_df,ddgs],ignore_index=True)

    return pdb1_df, pdb2_df, ddg_df

    

In [12]:
pdb1_df,pdb2_df,ddg_df=combining_ensembles(ens_dir_list=['../../dsci410_510/data/Fitzgerald_ens/ens_004/',
                                                         '../../dsci410_510/data/Fitzgerald_ens/ens_060/',
                                                         '../../dsci410_510/data/Fitzgerald_ens/ens_098/'])

In [13]:
pdb1_df

,site,resid,coord_array,x_coord,y_coord,z_coord,num_C,num_N,num_O,num_S,ens
0,1,10,"[[31.902, -1.036, -0.539]]",31.902,-1.036,-0.539,5,1,1,1,ens_004
1,2,15,"[[31.362, 2.073, 1.731]]",31.362,2.073,1.731,3,1,2,0,ens_004
2,3,6,"[[32.693, 2.55, 5.296]]",32.693,2.550,5.296,6,3,1,0,ens_004
3,4,6,"[[29.793, 1.99, 7.618]]",29.793,1.990,7.618,6,3,1,0,ens_004
4,5,18,"[[28.6, 0.403, 10.866]]",28.600,0.403,10.866,11,2,1,0,ens_004
...,...,...,...,...,...,...,...,...,...,...,...
916,210,15,"[[12.897, -7.278, 12.274]]",12.897,-7.278,12.274,3,1,2,0,ens_098
917,211,4,"[[11.444, -6.544, 15.716]]",11.444,-6.544,15.716,9,1,1,0,ens_098
918,212,12,"[[12.839, -6.351, 19.23]]",12.839,-6.351,19.230,5,1,1,0,ens_098
919,213,8,"[[12.029, -8.787, 22.08]]",12.029,-8.787,22.080,6,2,1,0,ens_098


In [14]:
pdb2_df

,site,resid,coord_array,x_coord,y_coord,z_coord,num_C,num_N,num_O,num_S,ens
0,1,10,"[[25.04, 13.829, 61.052]]",25.040,13.829,61.052,5,1,1,1,ens_004
1,2,15,"[[24.45, 17.585, 60.988]]",24.450,17.585,60.988,3,1,2,0,ens_004
2,3,6,"[[25.79, 19.178, 57.805]]",25.790,19.178,57.805,6,3,1,0,ens_004
3,4,6,"[[27.476, 22.478, 58.672]]",27.476,22.478,58.672,6,3,1,0,ens_004
4,5,18,"[[26.606, 25.151, 56.107]]",26.606,25.151,56.107,11,2,1,0,ens_004
...,...,...,...,...,...,...,...,...,...,...,...
916,210,15,"[[13.032, -7.171, 12.227]]",13.032,-7.171,12.227,3,1,2,0,ens_098
917,211,4,"[[11.481, -6.508, 15.674]]",11.481,-6.508,15.674,9,1,1,0,ens_098
918,212,12,"[[12.948, -6.285, 19.2]]",12.948,-6.285,19.200,5,1,1,0,ens_098
919,213,8,"[[12.12, -8.825, 21.961]]",12.120,-8.825,21.961,6,2,1,0,ens_098


In [15]:
ddg_df

,resid_num,wildtype,max_diff_ddg,ens
0,1.0,M,0.641229,ens_004
1,2.0,S,0.252171,ens_004
2,3.0,H,0.154710,ens_004
3,4.0,H,1.733517,ens_004
4,5.0,W,1.594987,ens_004
...,...,...,...,...
916,210.0,S,0.707968,ens_098
917,211.0,F,0.182805,ens_098
918,212.0,P,0.381759,ens_098
919,213.0,K,0.414916,ens_098


In [203]:
pdb1_df

,site,resid,coord_array,x_coord,y_coord,z_coord,num_C,num_N,num_O,num_S,ens
0,1,10,"[[31.902, -1.036, -0.539]]",31.902,-1.036,-0.539,5,1,1,1,ens_004
1,2,15,"[[31.362, 2.073, 1.731]]",31.362,2.073,1.731,3,1,2,0,ens_004
2,3,6,"[[32.693, 2.55, 5.296]]",32.693,2.550,5.296,6,3,1,0,ens_004
3,4,6,"[[29.793, 1.99, 7.618]]",29.793,1.990,7.618,6,3,1,0,ens_004
4,5,18,"[[28.6, 0.403, 10.866]]",28.600,0.403,10.866,11,2,1,0,ens_004
...,...,...,...,...,...,...,...,...,...,...,...
916,210,15,"[[12.897, -7.278, 12.274]]",12.897,-7.278,12.274,3,1,2,0,ens_098
917,211,4,"[[11.444, -6.544, 15.716]]",11.444,-6.544,15.716,9,1,1,0,ens_098
918,212,12,"[[12.839, -6.351, 19.23]]",12.839,-6.351,19.230,5,1,1,0,ens_098
919,213,8,"[[12.029, -8.787, 22.08]]",12.029,-8.787,22.080,6,2,1,0,ens_098


In [204]:
pdb1_df[:][['x_coord','y_coord','z_coord','num_C','num_N','num_O','num_S']].values

array([[31.902, -1.036, -0.539, ...,  1.   ,  1.   ,  1.   ],
       [31.362,  2.073,  1.731, ...,  1.   ,  2.   ,  0.   ],
       [32.693,  2.55 ,  5.296, ...,  3.   ,  1.   ,  0.   ],
       ...,
       [12.839, -6.351, 19.23 , ...,  1.   ,  1.   ,  0.   ],
       [12.029, -8.787, 22.08 , ...,  2.   ,  1.   ,  0.   ],
       [11.157, -7.076, 25.399, ...,  1.   ,  2.   ,  1.   ]])

In [48]:
#need to modify this function to get edge info for a bunch of graphs added together
#but need to make sure those nodes are not talking to each other
#example for one_hot_encoding of the residues
#y_one_hot = torch.nn.functional.one_hot(torch.Tensor(y).long(), num_classes=20)
def get_edge_info(df):

    edges=[]
    edge_features=[]
    for my_ens in df.ens.unique():
        print(my_ens)

        my_df=df.loc[df.ens == my_ens]

        start_idx=my_df.index[0]
        end_idx=my_df.index[-1]
        print(start_idx)
        print(end_idx)
    
        #pdb file coordinate numbers are angstroms
        #lets do all carbon alpha values within 10a of each other

        #0 for near each other, 1 for backbone bond
        for idx1,rows1 in my_df.iterrows():
        
            p1=my_df.loc[idx1,'coord_array']
        
            if idx1 == start_idx:
                edges.append([idx1,idx1+1])
                #edges = numpy.vstack([edges, [idx1,idx1+1]])
                edge_features.append(1)
        
            elif idx1 == end_idx:
                edges.append([idx1,idx1-1])
                #edges = numpy.vstack([edges, [idx1,idx1-1]])
                edge_features.append(1)
                
            else:
                edges.append([idx1, idx1-1])
                edges.append([idx1,idx1+1])
                #edges = numpy.vstack([edges, [[idx1,idx1-1],[idx1,idx1+1]]])
                edge_features.extend([1,1])
                
        
            for idx2,rows2 in my_df.iterrows():
        
                if idx1 != idx2:
        
                    p2=my_df.loc[idx2,'coord_array']
            
                    d = np.linalg.norm(p1 - p2)
        
                    if d<=10:
        
                        #do I assume that the nodes start at 0 and end at n number of nodes? 
                        #if so, I will use the index rather than the resid_num
                        #for now, will make that assumption
        
                        edges.append([idx1,idx2])
                        #edges = numpy.vstack([edges, [idx1,idx2]])
                        
                        edge_features.append(0)


    return edges, edge_features
            
        

In [63]:
def load_into_geom_data(df):

    mydata=Data()

    mydata.pos=torch.tensor(df[:][['x_coord','y_coord','z_coord']].values)
    mydata.x=torch.tensor(df[:][['num_C','num_N','num_O','num_S']].values)
    mydata.resid=torch.nn.functional.one_hot(torch.Tensor(df.resid).long(), num_classes=20)

    edges, edge_features=get_edge_info(df)

    mydata.edge_index=torch.tensor(edges)
    mydata.edge_attrs=torch.tensor(edge_features)

    return mydata
    

In [64]:
load_into_geom_data(pdb1_df)

ens_004
0
259
ens_060
260
706
ens_098
707
920


Data(pos=[921, 3], x=[921, 4], resid=[921, 20], edge_index=[19152, 2], edge_attrs=[19152])

In [40]:
myarray=np.empty([2,2])

In [41]:
myarray=np.vstack([myarray,[1,1]])

In [42]:
myarray

array([[1.10586413e-310, 1.10586413e-310],
       [1.10586413e-310, 1.10586413e-310],
       [1.00000000e+000, 1.00000000e+000]])

In [37]:
np.array(0)

array(0)

In [49]:
edges,edge_features=get_edge_info(pdb1_df)

ens_004
0
259
ens_060
260
706
ens_098
707
920


In [57]:
myt=torch.tensor((edges))
torch.transpose(myt, 0, 1)

tensor([[  0,   0,   0,  ..., 920, 920, 920],
        [  1,   1,   2,  ..., 917, 918, 919]])

In [193]:
pdb1_df = torch.nn.functional.one_hot(torch.Tensor(pdb1_df.resid).long(), num_classes=20)

In [191]:
y_one_hot

tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])

In [18]:
torch.tensor(pdb1_df[:][['num_C','num_N','num_O','num_S']].values)

tensor([[5, 1, 1, 1],
        [3, 1, 2, 0],
        [6, 3, 1, 0],
        ...,
        [5, 1, 1, 0],
        [6, 2, 1, 0],
        [5, 1, 2, 1]])

In [21]:
import torchvision.transforms as transforms
transform = transforms.Compose([
        transforms.ToTensor(), # Converts to tensor and 0->1
        transforms.Normalize((0.5,), (0.5)) ])
#^^this is what I've used before

In [17]:
torch.tensor(pdb1_df[:][['x_coord','y_coord','z_coord']].values)

tensor([[31.9020, -1.0360, -0.5390],
        [31.3620,  2.0730,  1.7310],
        [32.6930,  2.5500,  5.2960],
        ...,
        [12.8390, -6.3510, 19.2300],
        [12.0290, -8.7870, 22.0800],
        [11.1570, -7.0760, 25.3990]], dtype=torch.float64)

In [108]:
get_edge_info(pdb1_df)

ens_004
0
259
ens_060
260
706
ens_098
707
920


([(0, 1),
  (0, 1),
  (0, 2),
  (0, 3),
  (0, 230),
  (0, 231),
  (0, 234),
  (0, 235),
  (1, 0),
  (1, 2),
  (1, 0),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 230),
  (1, 231),
  (2, 1),
  (2, 3),
  (2, 0),
  (2, 1),
  (2, 3),
  (2, 4),
  (2, 5),
  (2, 10),
  (3, 2),
  (3, 4),
  (3, 0),
  (3, 1),
  (3, 2),
  (3, 4),
  (3, 5),
  (3, 6),
  (3, 10),
  (3, 11),
  (3, 62),
  (3, 63),
  (4, 3),
  (4, 5),
  (4, 1),
  (4, 2),
  (4, 3),
  (4, 5),
  (4, 6),
  (4, 7),
  (4, 9),
  (4, 10),
  (4, 11),
  (4, 12),
  (4, 14),
  (4, 15),
  (4, 18),
  (4, 62),
  (4, 63),
  (4, 198),
  (4, 199),
  (5, 4),
  (5, 6),
  (5, 2),
  (5, 3),
  (5, 4),
  (5, 6),
  (5, 7),
  (5, 8),
  (5, 9),
  (5, 10),
  (5, 11),
  (5, 12),
  (5, 14),
  (5, 15),
  (5, 62),
  (5, 63),
  (5, 198),
  (5, 199),
  (5, 229),
  (5, 242),
  (5, 243),
  (6, 5),
  (6, 7),
  (6, 3),
  (6, 4),
  (6, 5),
  (6, 7),
  (6, 8),
  (6, 9),
  (6, 10),
  (6, 11),
  (6, 12),
  (6, 13),
  (6, 14),
  (6, 15),
  (6, 63),
  (6, 198),
  (6, 229),
  (6, 239),
 

In [ ]:
#need to get node info and then will have function for everything
#then can just clean it up

In [189]:
data=Data()

In [16]:
from torch_geometric.data import Data